# UPAR 2027 Track 3: ViTPose-Large 384x288 STAGE 2 FINE-TUNING (Target: > 0.905 AP Top 1)
# Nhóm: VNUHCM-PCTDN
# Cải tiến Stage 2:
# 1. Learning Rate tinh chỉnh cực nhỏ: 1.5e-5 (Cosine Annealing)
# 2. OHKM khắt khe hơn: Top-6 khớp khó nhất (tập trung triệt tiêu sai số cổ tay, cổ chân, tai, đầu gối)
# 3. Adaptive Wing Loss cực nhọn: omega=16, alpha=2.2 định vị tâm khớp dưới 1 pixel
# 4. Đóng gói sẵn MULTI-SCALE FLIP TTA [1.15, 1.25, 1.35] + DARK GPU sub-pixel trong run.py


In [ ]:
!pip install -q transformers>=4.40.0 timm accelerate albumentations opencv-python-headless tqdm kagglehub


In [ ]:
# 2. Thiet lap thu muc lam viec va Clone repo chinh thuc de lay Annotations
import os, sys, json, shutil, zipfile, time
from pathlib import Path
import numpy as np, cv2, torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

WORKSPACE = Path('/kaggle/working')
DATA_DIR = Path('/kaggle/working/data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

REPO_DIR = Path('/kaggle/working/repo')
if not REPO_DIR.exists():
    print('[*] Cloning competition repository...')
    !git clone https://github.com/MickaelCormier/UPAR-Challenge-2027-Track3.git {REPO_DIR}
else:
    print('[*] Competition repo already exists.')

# Tu dong dong bo annotations sang DATA_DIR
if (REPO_DIR / 'data' / 'annotations').exists():
    shutil.copytree(REPO_DIR / 'data' / 'annotations', DATA_DIR / 'annotations', dirs_exist_ok=True)
    print('[+] Annotations da duoc dong bo san sang!')

sys.path.append(str(REPO_DIR))
print('[+] Workspace ready. PyTorch version:', torch.__version__, 'CUDA available:', torch.cuda.is_available())

In [ ]:
# 3. Tai va chuan bi bo du lieu UPAR (Bat buoc Full 100,000 anh PA-100K)
import os, sys, shutil, zipfile
from pathlib import Path
import gdown

WORKSPACE = Path('/kaggle/working')
DATA_DIR = Path('/kaggle/working/data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

print('[*] Dang chuan bi tron ven toan bo 42,736 mau cua Market-1501 va PA-100K...')

# ==========================================
# 1. CHUAN BI MARKET-1501 (12,923 mau)
# ==========================================
market_dst = DATA_DIR / 'market'
if not (market_dst / 'bounding_box_train').exists() or len(list((market_dst / 'bounding_box_train').glob('*.jpg'))) < 10000:
    print('[*] 1/2: Dang tai Market-1501 qua Google Drive truc tiep (153 MB)...')
    m_zip = DATA_DIR / 'market_1501.zip'
    if m_zip.exists() and m_zip.stat().st_size < 10 * 1024 * 1024:
        m_zip.unlink()
    
    try:
        gdown.download('https://drive.google.com/uc?id=0B8-rUzbwVRk0c054eEozWG9COHM', str(m_zip), quiet=False)
    except Exception as e:
        print(f'[!] Gdown loi: {e}')
        
    if m_zip.exists() and m_zip.stat().st_size > 50 * 1024 * 1024:
        print('[*] Giai nen Market-1501...')
        with zipfile.ZipFile(m_zip, 'r') as zf:
            zf.extractall(DATA_DIR)
        m_zip.unlink()
        
        for candidate in [DATA_DIR / 'Market-1501-v15.09.15', DATA_DIR / 'Market1501']:
            if candidate.exists():
                if market_dst.exists():
                    shutil.rmtree(market_dst)
                candidate.rename(market_dst)
                print(f'[+] Market-1501 da san sang tai: {market_dst}')
                break

# ==========================================
# 2. CHUAN BI PA-100K (Bat buoc >= 29,000 anh)
# ==========================================
pa100k_dst = DATA_DIR / 'pa100k' / 'images'
current_pa_count = len(list(pa100k_dst.glob('*.jpg'))) if pa100k_dst.exists() else 0

if current_pa_count < 25000:
    print(f'[*] PA-100K hien tai chi co {current_pa_count} anh (chua du 29,902 mau).')
    print('[*] 2/2: Dang tai PA-100K ban FULL 100,000 anh (451 MB) qua Google Drive...')
    if pa100k_dst.exists():
        shutil.rmtree(pa100k_dst)
        
    pa_zip = DATA_DIR / 'pa100k_data.zip'
    pa_raw_dir = DATA_DIR / 'pa100k_raw'
    pa_raw_dir.mkdir(parents=True, exist_ok=True)
    
    gdown.download('https://drive.google.com/uc?id=1Bod3MrbdCRiSvpR7NSOYi-rF3-ushwtj', str(pa_zip), quiet=False)
    
    print('[*] Giai nen PA-100K full...')
    with zipfile.ZipFile(pa_zip, 'r') as zf:
        zf.extractall(pa_raw_dir)
        
    for p in pa_raw_dir.rglob('*.jpg'):
        img_folder = p.parent
        pa100k_dst.parent.mkdir(parents=True, exist_ok=True)
        if pa100k_dst.exists():
            shutil.rmtree(pa100k_dst)
        img_folder.rename(pa100k_dst)
        print(f'[+] PA-100K full da duoc chuyen den: {pa100k_dst}')
        break
        
    if pa_zip.exists():
        pa_zip.unlink()
    if pa_raw_dir.exists():
        shutil.rmtree(pa_raw_dir)

market_count = len(list((market_dst / 'bounding_box_train').glob('*.jpg'))) if (market_dst / 'bounding_box_train').exists() else 0
pa_count = len(list(pa100k_dst.glob('*.jpg'))) if pa100k_dst.exists() else 0

print('\n' + '='*50)
print(f'[+] TONG HOP DU LIEU:')
print(f'    Market-1501: {market_count} anh (Chuan: ~12,936)')
print(f'    PA-100K:     {pa_count} anh (Chuan: ~100,000)')
print('='*50)

In [ ]:
# 4. Dinh nghia UPAR Dataset 384x288 voi Affine Transformation chuan xac tuyet doi & OHKM A-Wing Loss
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import json

# DO PHAN GIAI SIEU CAO CHO SINGLE MODEL
INP_H, INP_W = 384, 288
HM_H, HM_W = 96, 72

# POSETRACK18 SYMMETRY FIX:
# Khop 0: mui, Khop 1: co (head_bottom), Khop 2: dinh dau (head_top) nam tren truc doc chinh dien.
# TUYET DOI KHONG DUOC FLIP (1, 2) vi se lam lon nguoc dau va triet tieu AP_large!
FLIP_PAIRS = [(3, 4), (5, 6), (7, 8), (9, 10), (11, 12), (13, 14), (15, 16)]


def box_to_center_scale(box, out_w=288, out_h=384, normalize_factor=200.0, padding_factor=1.25):
    """
    Chuan hoa box thanh center va scale theo ty le khung hinh out_w/out_h = 288/384 (0.75).
    Khong bao gio lam meo ty le nguoi!
    """
    x, y, w, h = [float(v) for v in box[:4]]
    aspect = float(out_w) / float(out_h)
    center = np.array([x + w * 0.5, y + h * 0.5], dtype=np.float32)
    if w > aspect * h:
        h = w / aspect
    elif w < aspect * h:
        w = h * aspect
    scale = np.array([w / normalize_factor, h / normalize_factor], dtype=np.float32)
    return center, scale * padding_factor


def get_affine_transform(center, scale, rot, output_size):
    """
    Ma tran Affine chuan Top-Down Pose Estimation (MMPose / HRNet / ViTPose benchmark).
    Bao toan goc xoay, phong to/thu nho dong nhat (isotropic) va tu dong zero-pad khi tran vien.
    """
    scale_target = scale * 200.0
    src_w = scale_target[0]
    dst_w, dst_h = output_size
    rot_rad = np.pi * rot / 180.0
    src_dir = np.array([0.0, src_w * -0.5], dtype=np.float32)
    sn, cs = np.sin(rot_rad), np.cos(rot_rad)
    src_dir = np.array([src_dir[0] * cs - src_dir[1] * sn, src_dir[0] * sn + src_dir[1] * cs], dtype=np.float32)
    dst_dir = np.array([0.0, dst_w * -0.5], dtype=np.float32)

    src = np.zeros((3, 2), dtype=np.float32)
    dst = np.zeros((3, 2), dtype=np.float32)
    src[0, :] = center
    src[1, :] = center + src_dir
    src[2, :] = src[1, :] + np.array([-src_dir[1], src_dir[0]], dtype=np.float32)
    dst[0, :] = [dst_w * 0.5, dst_h * 0.5]
    dst[1, :] = np.array([dst_w * 0.5, dst_h * 0.5], dtype=np.float32) + dst_dir
    dst[2, :] = dst[1, :] + np.array([-dst_dir[1], dst_dir[0]], dtype=np.float32)
    return cv2.getAffineTransform(src, dst)


def generate_target_heatmaps(joints_3d, heatmap_size=(72, 96), sigma=3.0):
    """
    Tao target Gaussian Heatmap (96 x 72) cho 17 keypoints voi do chinh xac tieu chuan ViTPose.
    heatmap_size: (width, height) = (72, 96)
    sigma: 3.0 (tuong ung voi 2.0 o ban 64x48)
    """
    num_joints = joints_3d.shape[0]
    target = np.zeros((num_joints, heatmap_size[1], heatmap_size[0]), dtype=np.float32)
    mask = np.zeros((num_joints, 1, 1), dtype=np.float32)
    feat_stride = 4.0
    size = int(6 * sigma + 1)
    x = np.arange(0, size, 1, np.float32)
    y = x[:, np.newaxis]
    x0 = y0 = size // 2
    g = np.exp(- ((x - x0) ** 2 + (y - y0) ** 2) / (2 * sigma ** 2))

    for joint_id in range(num_joints):
        v = joints_3d[joint_id, 2]
        if v > 0:
            feat_x = int(joints_3d[joint_id, 0] / feat_stride + 0.5)
            feat_y = int(joints_3d[joint_id, 1] / feat_stride + 0.5)
            ul = [int(feat_x - 3 * sigma), int(feat_y - 3 * sigma)]
            br = [int(feat_x + 3 * sigma + 1), int(feat_y + 3 * sigma + 1)]
            if ul[0] >= heatmap_size[0] or ul[1] >= heatmap_size[1] or br[0] < 0 or br[1] < 0:
                continue
            g_x = max(0, -ul[0]), min(br[0], heatmap_size[0]) - ul[0]
            g_y = max(0, -ul[1]), min(br[1], heatmap_size[1]) - ul[1]
            img_x = max(0, ul[0]), min(br[0], heatmap_size[0])
            img_y = max(0, ul[1]), min(br[1], heatmap_size[1])
            target[joint_id, img_y[0]:img_y[1], img_x[0]:img_x[1]] = g[g_y[0]:g_y[1], g_x[0]:g_x[1]]
            mask[joint_id, 0, 0] = 1.0
    return target, mask


# VU KHI CUA DOI VO DICH: OHKM (Online Hard Keypoints Mining) + Adaptive Wing Loss
class OHKMAdaptiveWingLoss(nn.Module):
    def __init__(self, omega=16, theta=0.5, epsilon=1, alpha=2.2, top_k=6):
        super().__init__()
        self.omega = omega
        self.theta = theta
        self.epsilon = epsilon
        self.alpha = alpha
        self.top_k = top_k

    def forward(self, pred, target, mask=None):
        delta = (target - pred).abs()
        A = self.omega * (1.0 / (1.0 + (self.theta / self.epsilon) ** (self.alpha - target))) * (self.alpha - target) * ((self.theta / self.epsilon) ** (self.alpha - target - 1.0)) * (1.0 / self.epsilon)
        C = self.theta * A - self.omega * torch.log(1.0 + (self.theta / self.epsilon) ** (self.alpha - target))
        case1 = self.omega * torch.log(1.0 + (delta / self.epsilon) ** (self.alpha - target))
        case2 = A * delta - C
        loss = torch.where(delta < self.theta, case1, case2)

        # Tinh loss trung binh theo khong gian cho tung khop: (B, 17)
        kpt_loss = loss.mean(dim=[-2, -1])

        if mask is not None:
            m = mask.squeeze(-1).squeeze(-1)
            kpt_loss = kpt_loss * m
            num_valid = m.sum(dim=-1).clamp(min=1.0)
            k = min(self.top_k, 17)
            topk_losses, _ = torch.topk(kpt_loss, k=k, dim=-1)
            effective_k = torch.clamp(num_valid, max=k)
            sample_loss = topk_losses.sum(dim=-1) / effective_k
            return sample_loss.mean()
        else:
            topk_losses, _ = torch.topk(kpt_loss, k=min(self.top_k, 17), dim=-1)
            return topk_losses.mean()


class UPARPoseDataset(Dataset):
    def __init__(self, annotation_file, data_root, is_train=True):
        self.data_root = Path(data_root)
        self.is_train = is_train
        with open(annotation_file, 'r', encoding='utf-8') as f:
            data = json.load(f)

        id_to_img = {img['id']: img['file_name'] for img in data['images']}
        self.samples = []
        for ann in data['annotations']:
            box = ann.get('bbox', [])
            kpts = ann.get('keypoints', [])
            if len(box) == 4 and box[2] > 0 and box[3] > 0 and len(kpts) == 51:
                img_name = id_to_img.get(ann['image_id'])
                if img_name:
                    full_p = self.data_root / img_name
                    if full_p.exists():
                        self.samples.append({
                            'image_path': str(full_p),
                            'box': box,
                            'keypoints': np.array(kpts, dtype=np.float32).reshape(17, 3)
                        })
        print(f'[+] Loaded {len(self.samples):,} valid samples from {Path(annotation_file).name}')
        self.mean = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(1, 1, 3)
        self.std = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(1, 1, 3)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        img = cv2.imread(item['image_path'])
        if img is None:
            img = np.zeros((INP_H, INP_W, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        h_orig, w_orig = img.shape[:2]
        box = item['box'].copy()
        kpts = item['keypoints'].copy()

        # 1. Tinh toan Center va Scale theo ty le 288 x 384
        center, scale = box_to_center_scale(box, out_w=INP_W, out_h=INP_H, padding_factor=1.25)
        rot = 0.0

        if self.is_train:
            # Scale jitter
            scale = scale * np.random.uniform(0.85, 1.15)

            # Random rotation
            if np.random.rand() < 0.6:
                rot = np.random.uniform(-30, 30)

            # Horizontal Flip Augmentation voi FLIP_PAIRS chuan PoseTrack18
            if np.random.rand() < 0.5:
                img = cv2.flip(img, 1)
                center[0] = w_orig - center[0] - 1
                for k in range(17):
                    if kpts[k, 2] > 0:
                        kpts[k, 0] = w_orig - kpts[k, 0] - 1
                for l, r in FLIP_PAIRS:
                    kpts[l], kpts[r] = kpts[r].copy(), kpts[l].copy()

            # Blur mo phong camera giam sat kem chat luong
            if np.random.rand() < 0.35:
                ksize = np.random.choice([3, 5])
                img = cv2.GaussianBlur(img, (ksize, ksize), 0)

        # 2. Thuc hien Affine Transform chuan xac (KHONG BI MEO TI LE, KHONG BI MAT TRAN VIEN)
        trans = get_affine_transform(center, scale, rot, (INP_W, INP_H))
        cropped = cv2.warpAffine(img, trans, (INP_W, INP_H), flags=cv2.INTER_LINEAR)

        # 3. Surveillance Occlusion Augmentation (Che khuat gia lap nhu cot den, xe co, nguoi khac)
        if self.is_train and np.random.rand() < 0.5:
            num_cutouts = np.random.randint(1, 3)
            for _ in range(num_cutouts):
                eh = np.random.randint(25, 90)
                ew = np.random.randint(25, 90)
                ey = np.random.randint(0, INP_H - eh)
                ex = np.random.randint(0, INP_W - ew)
                cropped[ey:ey+eh, ex:ex+ew] = np.random.randint(0, 256, (3,), dtype=np.uint8)

        # Color Jitter nhe nhang mo phong dieu kien anh sang
        if self.is_train and np.random.rand() < 0.4:
            alpha = np.random.uniform(0.85, 1.15)
            beta = np.random.uniform(-15, 15)
            cropped = np.clip(alpha * cropped + beta, 0, 255).astype(np.uint8)

        # 4. Chuyen toa do keypoints sang khong gian anh crop 288x384
        warped_kpts = kpts.copy()
        for k in range(17):
            if warped_kpts[k, 2] > 0:
                pt = trans @ np.array([warped_kpts[k, 0], warped_kpts[k, 1], 1.0])
                warped_kpts[k, 0] = pt[0]
                warped_kpts[k, 1] = pt[1]
                if pt[0] < 0 or pt[0] >= INP_W or pt[1] < 0 or pt[1] >= INP_H:
                    warped_kpts[k, 2] = 0.0

        # 5. Sinh target heatmaps (96 x 72) voi sigma = 3.0
        target, mask = generate_target_heatmaps(warped_kpts, heatmap_size=(HM_W, HM_H), sigma=3.0)

        # 6. Chuan hoa tensor anh
        norm_img = ((cropped.astype(np.float32) / 255.0 - self.mean) / self.std).transpose(2, 0, 1)

        return torch.from_numpy(norm_img).float(), torch.from_numpy(target).float(), torch.from_numpy(mask).float()


print(f'[+] UPAR Dataset 384x288 (Heatmap: {HM_H}x{HM_W}) & OHKM Loss da duoc chuan hoa Affine Transform hoan hao!')


In [ ]:
# 5. Khoi tao ViTPose-Large 384x288 voi Bicubic Positional Embedding Interpolation
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import VitPoseConfig, VitPoseForPoseEstimation
from pathlib import Path

assert torch.cuda.is_available(), "[!] BẮT BUỘC BẬT GPU T4 x2 TRONG CỘT SESSION OPTIONS!"

num_gpus = torch.cuda.device_count()
DEVICE = torch.device('cuda:0')

MODEL_NAME = 'usyd-community/vitpose-plus-large'
print(f'[*] So luong GPU tim thay: {num_gpus} (NVIDIA T4 x2)')

# 1. Khoi tao Config 384x288
cfg = VitPoseConfig.from_pretrained(MODEL_NAME)
cfg.image_size = (384, 288)
cfg.backbone_config.image_size = (384, 288)

print('[*] Dang tao kien truc ViTPose-Large 384x288...')
model = VitPoseForPoseEstimation(cfg)

# 2. Load Pretrained weights (kiem tra neu co checkpoint v2 cu thi ke thua)
v2_path = WORKSPACE / 'fine_tuned_model_v2'
if v2_path.exists():
    print(f'[+] Phat hien checkpoint v2 (0.8752 AP) tai: {v2_path}')
    state_dict = torch.load(v2_path / 'pytorch_model.bin', map_location='cpu') if (v2_path / 'pytorch_model.bin').exists() else None
    if state_dict is None:
        from safetensors.torch import load_file
        state_dict = load_file(str(v2_path / 'model.safetensors'))
else:
    print(f'[*] Dang load weights tu checkpoint HuggingFace: {MODEL_NAME}...')
    pretrained_256 = VitPoseForPoseEstimation.from_pretrained(MODEL_NAME)
    state_dict = pretrained_256.state_dict()
    del pretrained_256

# 3. Noi suy Bicubic Positional Embeddings tu (16x12) -> (24x18)
pos_key = 'backbone.embeddings.position_embeddings'
if pos_key in state_dict:
    old_pos = state_dict[pos_key]
    if old_pos.shape[1] == 193:  # 16*12 + 1
        print('[*] Thuc hien noi suy Bicubic 2D Positional Embeddings tu (16x12) len (24x18)...')
        cls_token = old_pos[:, :1]
        pos_grid = old_pos[:, 1:].reshape(1, 16, 12, -1).permute(0, 3, 1, 2)
        new_grid = F.interpolate(pos_grid, size=(24, 18), mode='bicubic', align_corners=False)
        new_grid = new_grid.permute(0, 2, 3, 1).reshape(1, 432, -1)
        state_dict[pos_key] = torch.cat([cls_token, new_grid], dim=1)
        print(f'[+] Noi suy thanh cong: shape moi la {state_dict[pos_key].shape}')

# Nạp state_dict vào model 384x288
msg = model.load_state_dict(state_dict, strict=False)
print(f'[+] Model 384x288 da nap weights thanh cong! (Missing/Unexpected: {len(msg.missing_keys)}/{len(msg.unexpected_keys)})')

if num_gpus > 1:
    print(f'[+] Kich hoat Dual-GPU DataParallel tren {num_gpus} GPUs!')
    model = nn.DataParallel(model)
model = model.to(DEVICE)
print('[+] ViTPose-Large 384x288 da san sang cho huan luyen bứt phá 0.90+ AP!')

In [ ]:
# 6. Huan luyen ViTPose-Large 384x288 voi Dual-GPU, OHKM A-Wing Loss va AMP
import time
from torch.cuda.amp import autocast, GradScaler

ann_file = REPO_DIR / 'data' / 'annotations' / 'specialization' / 'train_upar_all.json'
if not ann_file.exists():
    ann_file = DATA_DIR / 'annotations' / 'specialization' / 'train_upar_all.json'

print(f'[*] Loading annotations tu: {ann_file}')
train_dataset = UPARPoseDataset(ann_file, DATA_DIR, is_train=True)
print(f'[+] TONG CONG: {len(train_dataset):,} mau tren camera giam sat!')

# Voi 384x288, Batch Size = 32 (16 per T4 GPU) chiem ~10GB VRAM moi GPU (cuc ky an toan)
num_gpus = torch.cuda.device_count()
BATCH_SIZE = 32 if num_gpus > 1 else 16
NUM_WORKERS = 4
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True
)

criterion = OHKMAdaptiveWingLoss(omega=16, theta=0.5, epsilon=1, alpha=2.2, top_k=6)

# LR toi uu cho Warm-start Fine-tuning: 4e-5
optimizer = torch.optim.AdamW(model.parameters(), lr=1.5e-5, weight_decay=1e-4)
EPOCHS = 4
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS * len(train_loader), eta_min=1e-6)
scaler = GradScaler()

SAVE_DIR = WORKSPACE / 'fine_tuned_model_large_384'
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print(f'[*] Bat dau huan luyen {EPOCHS} epochs ViTPose-Large 384x288 tren Dual T4 GPUs...')
start_all = time.time()

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()
    model.train()
    running_loss = 0.0
    
    for step, (images, targets, masks) in enumerate(train_loader):
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        
        ds_idx = torch.zeros(images.shape[0], dtype=torch.long, device=DEVICE)
        with autocast():
            outputs = model(pixel_values=images, dataset_index=ds_idx)
            pred_heatmaps = outputs.heatmaps if hasattr(outputs, 'heatmaps') else outputs
            loss = criterion(pred_heatmaps, targets, masks)
            
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        running_loss += loss.item()
        
        if (step + 1) % 100 == 0 or (step + 1) == len(train_loader):
            lr_cur = optimizer.param_groups[0]['lr']
            print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] Step [{step+1:04d}/{len(train_loader)}] "
                  f"Loss: {loss.item():.5f} (Avg: {running_loss/(step+1):.5f}) LR: {lr_cur:.6f}")
                  
    epoch_time = (time.time() - epoch_start) / 60.0
    print(f"[+] Hoan tat Epoch {epoch:02d} trong {epoch_time:.1f} phut! Avg Loss: {running_loss/len(train_loader):.5f}")
    raw_model = model.module if hasattr(model, "module") else model
    raw_model.save_pretrained(str(SAVE_DIR))
    print(f"    [+] Da luu checkpoint sau Epoch {epoch:02d} tai: {SAVE_DIR}")

total_time = (time.time() - start_all) / 60.0
print(f'[*] Huan luyen 384x288 hoan tat trong {total_time:.1f} phut!')

raw_model = model.module if hasattr(model, 'module') else model
raw_model.save_pretrained(str(SAVE_DIR))
print(f'[+] Checkpoint ViTPose-Large 384x288 da luu tai: {SAVE_DIR}')

In [ ]:
# 7. Dong goi Model thanh TorchScript 384x288 cho Container danh gia
SUBMISSION_DIR = WORKSPACE / 'submission'
ASSETS_DIR = SUBMISSION_DIR / 'assets'
ASSETS_DIR.mkdir(parents=True, exist_ok=True)

raw_model = model.module if hasattr(model, 'module') else model

class CocoWrapper(nn.Module):
    def __init__(self, vit_model, dataset_index=0):
        super().__init__()
        self.model = vit_model
        self.register_buffer('dataset_index', torch.tensor([dataset_index], dtype=torch.int64))

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        index = self.dataset_index.to(pixel_values.device).expand(pixel_values.shape[0])
        return self.model(pixel_values=pixel_values, dataset_index=index).heatmaps

print('[*] Tracing TorchScript model 384x288 tren CPU...')
model_cpu = raw_model.cpu().eval()
wrapper = CocoWrapper(model_cpu, 0).eval()
example = torch.zeros(1, 3, 384, 288)

with torch.no_grad():
    traced = torch.jit.trace(wrapper, example, strict=False, check_trace=False)
    traced = torch.jit.freeze(traced)
    torchscript_path = ASSETS_DIR / 'vitpose.torchscript'
    traced.save(str(torchscript_path))

print(f'[+] Saved {torchscript_path} ({torchscript_path.stat().st_size / 1e6:.1f} MB)')

meta = {
    'source_checkpoint': 'vitpose-large-384x288',
    'dataset_index': 0,
    'input_height': 384,
    'input_width': 288,
    'image_mean': [0.485, 0.456, 0.406],
    'image_std': [0.229, 0.224, 0.225],
    'rescale_factor': 0.00392156862745098,
    'normalize_factor': 200.0,
    'padding_factor': 1.25,
    'num_keypoints': 17,
    'heatmap_scale_factor': 4
}
with open(ASSETS_DIR / 'preprocess.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2)

shutil.copy2(REPO_DIR / 'examples' / 'task3' / 'vitpose_submission' / 'assets' / 'mean_pose.json', ASSETS_DIR / 'mean_pose.json')
print('[+] Assets metadata 384x288 saved successfully.')

In [ ]:
# 8. Tao file run.py chuan xac tuyet doi voi MULTI-SCALE FLIP TTA [1.15, 1.25, 1.35] + DARK GPU
import shutil

SUBMISSION_DIR = WORKSPACE / 'submission'
REPO_DIR = Path('/kaggle/working/repo')
if not REPO_DIR.exists():
    REPO_DIR = Path('/kaggle/working/UPAR-Challenge-2027-Track3')

RUN_PY_CONTENT = r"""from __future__ import annotations

import json
import os
from typing import Any

import numpy as np
import torch
import torch.nn.functional as F
torch.set_num_threads(4)
from PIL import Image

HERE = os.path.dirname(os.path.abspath(__file__))
MODEL_PATH = os.path.join(HERE, "assets", "vitpose.torchscript")
META_PATH = os.path.join(HERE, "assets", "preprocess.json")
PRIOR_PATH = os.path.join(HERE, "assets", "mean_pose.json")

# Batched inference toi uu cho GPU Codabench (Nvidia T4)
BATCH_SIZE = 32

# MULTI-SCALE TTA SCALES:
# 1.15: Zoom-in (phong to de bat chi tiet khop nho: mat, mui, tai, co tay, co chan)
# 1.25: Standard baseline (chuan muc dao tao)
# 1.35: Zoom-out (bao quat nguoi cao, sai tay/chan rong, tu the dac biet)
PADDING_SCALES = [1.15, 1.25, 1.35]

_MODEL = None
_DEVICE = None
_META: dict[str, Any] = {}
_PRIOR: list[list[float]] | None = None
_MEAN: np.ndarray | None = None
_STD: np.ndarray | None = None

# PoseTrack18 keypoints: 0(mui), 1(co), 2(dinh dau) nam tren truc doc, KHONG DOI!
FLIP_PAIRS = [(3, 4), (5, 6), (7, 8), (9, 10), (11, 12), (13, 14), (15, 16)]


def load_model() -> None:
    global _MODEL, _DEVICE, _META, _PRIOR, _MEAN, _STD
    if _MODEL is not None:
        return

    if not os.path.isfile(MODEL_PATH):
        raise FileNotFoundError(f"{MODEL_PATH} is missing.")

    with open(META_PATH, "r", encoding="utf-8") as fh:
        _META = json.load(fh)
    with open(PRIOR_PATH, "r", encoding="utf-8") as fh:
        _PRIOR = json.load(fh)["mean_pose_normalized"]

    _MEAN = np.asarray(_META["image_mean"], dtype=np.float32).reshape(3, 1, 1)
    _STD = np.asarray(_META["image_std"], dtype=np.float32).reshape(3, 1, 1)

    _DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.set_grad_enabled(False)
    _MODEL = torch.jit.load(MODEL_PATH, map_location=_DEVICE)
    _MODEL.eval()
    print(f"[submission] ViTPose-Large 384x288 ready on {_DEVICE} with Multi-Scale Flip TTA ({_META['input_width']}x{_META['input_height']})", flush=True)


def box_to_center_scale(box, out_w: int, out_h: int, normalize_factor: float, padding_factor: float):
    x, y, w, h = [float(v) for v in box[:4]]
    aspect = out_w / out_h
    center = np.array([x + w * 0.5, y + h * 0.5], dtype=np.float32)
    if w > aspect * h:
        h = w / aspect
    elif w < aspect * h:
        w = h * aspect
    scale = np.array([w / normalize_factor, h / normalize_factor], dtype=np.float32)
    return center, scale * padding_factor


def warp_matrix(center: np.ndarray, scale: np.ndarray, out_w: int, out_h: int) -> np.ndarray:
    size_input = center * 2.0
    size_dst = np.array([out_w, out_h], dtype=np.float32) - 1.0
    size_target = scale * 200.0
    scale_x = size_dst[0] / size_target[0]
    scale_y = size_dst[1] / size_target[1]
    matrix = np.zeros((2, 3), dtype=np.float32)
    matrix[0, 0] = scale_x
    matrix[0, 2] = scale_x * (-0.5 * size_input[0] + 0.5 * size_target[0])
    matrix[1, 1] = scale_y
    matrix[1, 2] = scale_y * (-0.5 * size_input[1] + 0.5 * size_target[1])
    return matrix


def warp_affine(image: np.ndarray, matrix: np.ndarray, out_h: int, out_w: int) -> np.ndarray:
    bottom = np.array([[0.0, 0.0, 1.0]], dtype=np.float32)
    inverse = np.linalg.inv(np.vstack([matrix, bottom]))[:2].astype(np.float32)
    ys, xs = np.meshgrid(np.arange(out_h, dtype=np.float32), np.arange(out_w, dtype=np.float32), indexing="ij")
    src_x = inverse[0, 0] * xs + inverse[0, 1] * ys + inverse[0, 2]
    src_y = inverse[1, 0] * xs + inverse[1, 1] * ys + inverse[1, 2]

    height, width = image.shape[:2]
    x0 = np.floor(src_x).astype(np.int64)
    y0 = np.floor(src_y).astype(np.int64)
    x1, y1 = x0 + 1, y0 + 1
    wx, wy = src_x - x0, src_y - y0

    inside = (src_x >= 0) & (src_x <= width - 1) & (src_y >= 0) & (src_y <= height - 1)
    x0c, x1c = np.clip(x0, 0, width - 1), np.clip(x1, 0, width - 1)
    y0c, y1c = np.clip(y0, 0, height - 1), np.clip(y1, 0, height - 1)

    top = image[y0c, x0c] * (1 - wx)[..., None] + image[y0c, x1c] * wx[..., None]
    bottom = image[y1c, x0c] * (1 - wx)[..., None] + image[y1c, x1c] * wx[..., None]
    return (top * (1 - wy)[..., None] + bottom * wy[..., None]) * inside[..., None]


def preprocess(image: np.ndarray, box, pad_factor: float) -> tuple:
    out_w, out_h = int(_META["input_width"]), int(_META["input_height"])
    center, scale = box_to_center_scale(box, out_w, out_h, float(_META["normalize_factor"]), pad_factor)
    crop = warp_affine(image, warp_matrix(center, scale, out_w, out_h), out_h, out_w)
    tensor = crop.transpose(2, 0, 1) * float(_META["rescale_factor"])
    return (((tensor - _MEAN) / _STD).astype(np.float32), center, scale)


def decode_heatmaps_gpu(heatmaps: torch.Tensor, centers: np.ndarray, scales: np.ndarray):
    batch, num_kpts, height, width = heatmaps.shape
    flat = heatmaps.view(batch, num_kpts, -1)
    scores, idx = torch.max(flat, dim=2)

    coords_x = (idx % width).float()
    coords_y = (idx // width).float()

    padded = F.pad(heatmaps, (1, 1, 1, 1), mode="replicate")
    
    b_idx = torch.arange(batch, device=heatmaps.device)[:, None]
    k_idx = torch.arange(num_kpts, device=heatmaps.device)[None, :]
    px = coords_x.long() + 1
    py = coords_y.long() + 1

    dx = (padded[b_idx, k_idx, py, px + 1] - padded[b_idx, k_idx, py, px - 1]) * 0.5
    dy = (padded[b_idx, k_idx, py + 1, px] - padded[b_idx, k_idx, py - 1, px]) * 0.5
    dxx = padded[b_idx, k_idx, py, px + 1] - 2 * padded[b_idx, k_idx, py, px] + padded[b_idx, k_idx, py, px - 1]
    dyy = padded[b_idx, k_idx, py + 1, px] - 2 * padded[b_idx, k_idx, py, px] + padded[b_idx, k_idx, py - 1, px]

    offset_x = torch.clamp(dx / (dxx.abs() + 1e-6), -0.5, 0.5)
    offset_y = torch.clamp(dy / (dyy.abs() + 1e-6), -0.5, 0.5)
    offset_x = torch.nan_to_num(offset_x, nan=0.0)
    offset_y = torch.nan_to_num(offset_y, nan=0.0)

    final_x = (coords_x + offset_x).cpu().numpy()
    final_y = (coords_y + offset_y).cpu().numpy()
    scores_np = scores.cpu().numpy()

    out = np.zeros((batch, num_kpts, 2), dtype=np.float32)
    for i in range(batch):
        scale = scales[i] * 200.0
        out[i, :, 0] = final_x[i] * scale[0] / (width - 1.0) + centers[i][0] - scale[0] * 0.5
        out[i, :, 1] = final_y[i] * scale[1] / (height - 1.0) + centers[i][1] - scale[1] * 0.5
    return out, scores_np


def _prior_pose(box) -> list[list[float]]:
    bx, by, bw, bh = [float(v) for v in box]
    return [[bx + u * bw, by + v * bh, 1.0] for u, v in _PRIOR]


def _resolve_image_path(sample: dict[str, Any]) -> str:
    image_path = sample.get("image_path")
    if image_path and os.path.isfile(image_path):
        return image_path
    file_name = sample.get("file_name")
    if file_name and os.path.isfile(file_name):
        return file_name
    raise FileNotFoundError("Could not resolve image path for sample.")


def predict_image(sample: dict[str, Any]) -> dict[str, Any]:
    if _MODEL is None:
        load_model()

    boxes = sample["boxes"]
    if not boxes:
        return {"keypoints": [], "scores": []}

    image = np.asarray(Image.open(_resolve_image_path(sample)).convert("RGB"), dtype=np.float32)

    keypoints: list[Any] = [None] * len(boxes)
    scores: list[float] = [0.0] * len(boxes)

    usable = [i for i, b in enumerate(boxes) if float(b[2]) > 0 and float(b[3]) > 0]
    for i in set(range(len(boxes))) - set(usable):
        keypoints[i] = _prior_pose(boxes[i])
        scores[i] = 0.01

    for start in range(0, len(usable), BATCH_SIZE):
        chunk = usable[start:start + BATCH_SIZE]
        
        all_scale_coords = []
        all_scale_scores = []
        
        # Multi-Scale Loop: chay qua ca 3 ty le zoom-in (1.15), standard (1.25), zoom-out (1.35)
        for pad_factor in PADDING_SCALES:
            tensors, centers, scales = [], [], []
            for i in chunk:
                tensor, center, scale = preprocess(image, boxes[i], pad_factor)
                tensors.append(tensor)
                centers.append(center)
                scales.append(scale)

            batch = torch.from_numpy(np.stack(tensors)).to(_DEVICE)

            # 1. Du doan huong goc (384x288)
            heatmaps = _MODEL(batch).float()

            # 2. Du doan Flip TTA chuan PoseTrack18 (KHONG hoan doi co/dinh dau 1, 2)
            flipped_batch = torch.flip(batch, dims=[-1])
            flipped_heatmaps = _MODEL(flipped_batch).float()
            flipped_heatmaps = torch.flip(flipped_heatmaps, dims=[-1])
            for left, right in FLIP_PAIRS:
                tmp = flipped_heatmaps[:, left].clone()
                flipped_heatmaps[:, left] = flipped_heatmaps[:, right]
                flipped_heatmaps[:, right] = tmp

            # 3. Hop nhat Heatmap cho scale nay
            combined_heatmaps = (heatmaps + flipped_heatmaps) * 0.5

            coords, joint_scores = decode_heatmaps_gpu(combined_heatmaps, np.stack(centers), np.stack(scales))
            all_scale_coords.append(coords)
            all_scale_scores.append(joint_scores)

        # 4. Trung binh ket qua tu tat ca cac Scale (Multi-Scale Fusion)
        avg_coords = np.mean(all_scale_coords, axis=0)
        avg_joint_scores = np.mean(all_scale_scores, axis=0)

        for j, i in enumerate(chunk):
            keypoints[i] = [[float(x), float(y), float(avg_joint_scores[j][k])] for k, (x, y) in enumerate(avg_coords[j])]
            scores[i] = float(np.mean(np.sort(avg_joint_scores[j])[-8:]))

    return {"keypoints": keypoints, "scores": scores}
"""

with open(SUBMISSION_DIR / 'run.py', 'w', encoding='utf-8') as f:
    f.write(RUN_PY_CONTENT.strip() + '\n')

meta_yaml = """# Code submissions are executed by the challenges ingestion program, which
# imports run.py and calls predict_image(). Nothing in this file is used for
# inference, it exists only so the platform never treats the archive as a
# result submission. Leave it as-is.
command: python3 -c "print('Submission package ready. The ingestion program imports run.py')"
"""
with open(SUBMISSION_DIR / 'metadata.yaml', 'w', encoding='utf-8') as f:
    f.write(meta_yaml)

req_txt = """torch>=2.0
numpy>=1.24
pillow>=9.0
"""
with open(SUBMISSION_DIR / 'requirements.txt', 'w', encoding='utf-8') as f:
    f.write(req_txt)

print('[+] run.py Multi-Scale Flip TTA [1.15, 1.25, 1.35] + metadata.yaml + requirements.txt da tao xong!')


In [ ]:
# 9. Dong goi file submission.zip va Tao Link Download Truc Tiep
from IPython.display import FileLink, display

zip_path = WORKSPACE / 'submission.zip'
if zip_path.exists():
    zip_path.unlink()

print('[*] Zipping submission folder...')
!cd {SUBMISSION_DIR} && zip -r {zip_path} .

print(f'[+] Submission created: {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)')

print('[*] Verifying submission.zip CRC integrity...')
with zipfile.ZipFile(zip_path, 'r') as zf:
    bad_file = zf.testzip()
    assert bad_file is None, f'Zip integrity test failed on {bad_file}'
    print('[+] All files inside submission.zip are 100% verified!')
    print('[+] Files list:', zf.namelist())

print('\n' + '='*60)
print('[+] BẤM VÀO ĐƯỜNG LINK XANH DƯỚI ĐÂY ĐỂ TẢI FILE VỀ MÁY:')
print('='*60)
display(FileLink('submission.zip'))

## HOAN TAT!
File **submission.zip** da duoc tao thanh cong tai:
/kaggle/working/submission.zip

### Cach tai ve may:
1. Nhin sang thanh cong cu ben phai (muc **Output**).
2. Tim file **submission.zip**, bam vao bieu tuong **3 cham -> Download**.
3. Nop file vua tai len [Codabench UPAR Track 3](https://www.codabench.org/competitions/18273/).

Chuc ban gianh diem so top dau!